In [1]:
print("Hello")

Hello


Great 👍 now we move to **Reinforcement Learning (RL)**.

Reinforcement Learning =
An **agent** learns by interacting with an **environment**, receiving **rewards**, and improving its policy over time.

Very useful in:

* Network traffic engineering
* Routing optimization
* Congestion control
* Autonomous scaling

---

# 🔥 1️⃣ Q-Learning (Tabular RL)

Used when state/action space is small.

### 📌 Example: Simple Grid Environment

```python
import numpy as np
import gymnasium as gym

# Create environment
env = gym.make("FrozenLake-v1", is_slippery=False)

# Initialize Q-table
state_size = env.observation_space.n
action_size = env.action_space.n
q_table = np.zeros((state_size, action_size))

# Hyperparameters
alpha = 0.8       # Learning rate
gamma = 0.95      # Discount factor
epsilon = 0.1     # Exploration rate
episodes = 1000

for episode in range(episodes):
    state, _ = env.reset()
    done = False

    while not done:
        if np.random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        next_state, reward, done, _, _ = env.step(action)

        # Q-learning update rule
        q_table[state, action] = q_table[state, action] + alpha * (
            reward + gamma * np.max(q_table[next_state]) - q_table[state, action]
        )

        state = next_state

print("Trained Q-table:")
print(q_table)
```

👉 Used in routing path optimization (small topologies).

---

# 🔥 2️⃣ Deep Q Network (DQN)

When state space is large → use neural network instead of Q-table.

```python
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

env = gym.make("CartPole-v1")

# Neural Network
class DQN(nn.Module):
    def __init__(self):
        super(DQN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(4, 24),
            nn.ReLU(),
            nn.Linear(24, 2)
        )

    def forward(self, x):
        return self.fc(x)

model = DQN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

episodes = 500
gamma = 0.95

for episode in range(episodes):
    state, _ = env.reset()
    state = torch.FloatTensor(state)
    done = False

    while not done:
        q_values = model(state)
        action = torch.argmax(q_values).item()

        next_state, reward, done, _, _ = env.step(action)
        next_state = torch.FloatTensor(next_state)

        target = reward + gamma * torch.max(model(next_state))
        loss = loss_fn(q_values[action], target.detach())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state

print("DQN Training Complete")
```

👉 Used in:

* Traffic engineering
* Adaptive congestion control
* SDN path selection

---

# 🔥 3️⃣ Policy Gradient (REINFORCE)

Instead of learning Q-values, directly learns policy.

```python
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim

env = gym.make("CartPole-v1")

class PolicyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(4, 24),
            nn.ReLU(),
            nn.Linear(24, 2),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        return self.fc(x)

policy = PolicyNetwork()
optimizer = optim.Adam(policy.parameters(), lr=0.01)

gamma = 0.99
episodes = 500

for episode in range(episodes):
    state, _ = env.reset()
    state = torch.FloatTensor(state)
    log_probs = []
    rewards = []
    done = False

    while not done:
        probs = policy(state)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_probs.append(dist.log_prob(action))

        next_state, reward, done, _, _ = env.step(action.item())
        rewards.append(reward)
        state = torch.FloatTensor(next_state)

    # Compute discounted rewards
    discounted = []
    cumulative = 0
    for r in reversed(rewards):
        cumulative = r + gamma * cumulative
        discounted.insert(0, cumulative)

    discounted = torch.FloatTensor(discounted)

    loss = -sum(lp * G for lp, G in zip(log_probs, discounted))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Policy Gradient Training Complete")
```

👉 Used in:

* Dynamic bandwidth allocation
* Cloud resource scaling
* Autonomous routing

---

# 🔥 4️⃣ SARSA (On-policy RL)

Similar to Q-learning but updates using current policy.

```python
import numpy as np
import gymnasium as gym

env = gym.make("FrozenLake-v1", is_slippery=False)

q_table = np.zeros((env.observation_space.n, env.action_space.n))
alpha = 0.8
gamma = 0.95
epsilon = 0.1
episodes = 1000

for episode in range(episodes):
    state, _ = env.reset()
    if np.random.rand() < epsilon:
        action = env.action_space.sample()
    else:
        action = np.argmax(q_table[state])

    done = False
    while not done:
        next_state, reward, done, _, _ = env.step(action)

        if np.random.rand() < epsilon:
            next_action = env.action_space.sample()
        else:
            next_action = np.argmax(q_table[next_state])

        q_table[state, action] += alpha * (
            reward + gamma * q_table[next_state, next_action] - q_table[state, action]
        )

        state = next_state
        action = next_action

print("SARSA Training Complete")
```

---

# 🔥 5️⃣ Actor-Critic (Advanced RL)

Combines:

* Actor (policy)
* Critic (value function)

Used in:

* SDN controllers
* Traffic flow optimization
* Real-time decision systems

(Full implementation is longer; commonly implemented using Stable-Baselines3.)

---

# 🧠 Real Network Engineering Use Cases

| RL Algorithm    | Network Use Case                 |
| --------------- | -------------------------------- |
| Q-Learning      | Path selection in small topology |
| DQN             | Congestion control in large DC   |
| Policy Gradient | Dynamic load balancing           |
| Actor-Critic    | Traffic engineering              |
| Multi-Agent RL  | ISP backbone optimization        |

---

# 🚀 Why RL Is Powerful for Network Engineers

Unlike supervised ML:

* No labeled data needed
* Learns by interacting
* Optimizes long-term reward
* Perfect for routing and traffic engineering

---

If you want next, I can:

* 🔥 Show RL example specifically for routing optimization
* 🔥 Build a mini SDN simulation example
* 🔥 Show how RL can replace OSPF cost tuning
* 🔥 Give interview-focused RL explanations

What direction do you want?


In [2]:
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 10.3 MB/s  0:00:01m0:00:0100:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 20.0 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 35.0 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
!pip install gymnasium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 7.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gymnasium]/3 [gymnasium]e]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
!python3 -m pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.2 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3


Q-Learning (Tabular RL)

Used when state/action space is small.

📌 Example: Simple Grid Environment

In [6]:
import numpy as np
import gymnasium as gym

# Create environment
env = gym.make("FrozenLake-v1", is_slippery=False)

# Initialize Q-table
state_size = env.observation_space.n
action_size = env.action_space.n
q_table = np.zeros((state_size, action_size))

# Hyperparameters
alpha = 0.8       # Learning rate
gamma = 0.95      # Discount factor
epsilon = 0.1     # Exploration rate
episodes = 1000

for episode in range(episodes):
    state, _ = env.reset()
    done = False

    while not done:
        if np.random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        next_state, reward, done, _, _ = env.step(action)

        # Q-learning update rule
        q_table[state, action] = q_table[state, action] + alpha * (
            reward + gamma * np.max(q_table[next_state]) - q_table[state, action]
        )

        state = next_state

print("Trained Q-table:")
print(q_table)

Trained Q-table:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


2️⃣ Deep Q Network (DQN)

When state space is large → use neural network instead of Q-table.

In [7]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 13.5 MB/s  0:00:24m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 52.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 21.1 MB/s  0:00:14m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 33.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 48.8 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 28.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 20.2 MB/s  0:00:15m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 47.3 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 58.4 MB/s  0:00:016m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 49.3 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.

In [13]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

env = gym.make("CartPole-v1")

# Neural Network
class DQN(nn.Module):
    def __init__(self):
        super(DQN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(4, 24),
            nn.ReLU(),
            nn.Linear(24, 2)
        )

    def forward(self, x):
        return self.fc(x)

model = DQN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

episodes = 500
gamma = 0.95

for episode in range(episodes):
    state, _ = env.reset()
    state = torch.FloatTensor(state)
    done = False

    while not done:
        q_values = model(state)
        action = torch.argmax(q_values).item()

        next_state, reward, done, _, _ = env.step(action)
        next_state = torch.FloatTensor(next_state)

        target = reward + gamma * torch.max(model(next_state))
        loss = loss_fn(q_values[action], target.detach())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state

print("DQN Training Complete")

DQN Training Complete


Perfect 👌 this is exactly the kind of advanced topic that stands out in Amazon NDE / hyperscaler interviews.

We’ll do two things:

1️⃣ RL for Routing Optimization (path selection)
2️⃣ How RL can replace static OSPF cost tuning

---

# 🔥 PART 1 — RL for Routing Optimization

## 🎯 Problem

Traditional routing (OSPF, IS-IS):

* Uses static link cost
* Optimizes shortest path only
* Does NOT dynamically adapt to:

  * Congestion
  * Latency spikes
  * Packet loss
  * Link utilization

We want:

> Agent learns best path dynamically based on real-time network state.

---

## 🧠 Concept Mapping

| RL Concept | Network Meaning                    |
| ---------- | ---------------------------------- |
| State      | Current link utilization / delay   |
| Action     | Choose next-hop path               |
| Reward     | Low latency + low congestion       |
| Episode    | One packet flow / routing decision |

---

## 🔥 Example: RL Path Selection

We simulate a small topology:

```
A ---- B ---- D
 \             /
  ---- C -----
```

Two possible paths from A → D:

* Path1: A-B-D
* Path2: A-C-D

We want RL to learn which path is better based on congestion.

---

## 🧪 Python Example (Q-Learning for Routing)

```python
import numpy as np
import random

# States represent congestion level
# 0 = low congestion
# 1 = medium congestion
# 2 = high congestion
states = [0, 1, 2]

# Actions: 0 = Path A-B-D, 1 = Path A-C-D
actions = [0, 1]

# Initialize Q-table
q_table = np.zeros((len(states), len(actions)))

alpha = 0.8
gamma = 0.9
epsilon = 0.1
episodes = 1000

for episode in range(episodes):

    state = random.choice(states)

    # epsilon-greedy
    if random.uniform(0,1) < epsilon:
        action = random.choice(actions)
    else:
        action = np.argmax(q_table[state])

    # Simulated reward logic
    # Suppose path 0 performs badly during congestion
    if state == 2 and action == 0:
        reward = -10
    else:
        reward = 10 - state*2

    next_state = random.choice(states)

    # Q-learning update
    q_table[state, action] = q_table[state, action] + alpha * (
        reward + gamma * np.max(q_table[next_state]) - q_table[state, action]
    )

print("Learned Q-table:")
print(q_table)
```

---

## 🎯 Result

After training:

* During high congestion → agent prefers alternate path
* During low congestion → shortest path preferred

This mimics:

* Dynamic traffic engineering
* Load balancing
* ECMP optimization

---

# 🔥 PART 2 — Replacing OSPF Cost Tuning with RL

## 🚨 Traditional OSPF Problem

OSPF metric = static cost based on:

```
Cost = Reference Bandwidth / Interface Bandwidth
```

Problems:

* Doesn’t consider:

  * Real-time latency
  * Packet loss
  * Congestion
* Manual cost tuning required
* Slow convergence to optimal path

---

## 🔥 RL-Based OSPF Cost Replacement

Instead of:

```
interface Gig0/0
 ip ospf cost 10
```

We let RL dynamically decide cost.

---

## 🧠 How It Works

Each link cost becomes:

```
Dynamic Cost = f(utilization, delay, loss)
```

RL agent:

* Observes telemetry (streaming telemetry / NetFlow)
* Adjusts virtual link weight
* SDN controller pushes updated weights

---

## 🔥 Example: RL Adjusting Link Cost

```python
import numpy as np

# State: [utilization %, latency ms]
# Action: increase cost or decrease cost

q_table = np.zeros((3, 2))  # 3 utilization states, 2 actions

def get_state(util):
    if util < 40:
        return 0
    elif util < 70:
        return 1
    else:
        return 2

alpha = 0.7
gamma = 0.9
epsilon = 0.1

for _ in range(1000):
    utilization = np.random.randint(0,100)
    state = get_state(utilization)

    if np.random.rand() < epsilon:
        action = np.random.choice([0,1])
    else:
        action = np.argmax(q_table[state])

    # Reward logic
    if state == 2 and action == 0:  # high util but didn't increase cost
        reward = -20
    else:
        reward = 10

    next_state = get_state(np.random.randint(0,100))

    q_table[state, action] += alpha * (
        reward + gamma*np.max(q_table[next_state]) - q_table[state, action]
    )

print("Optimized OSPF Cost Policy:")
print(q_table)
```

---

## 🔥 Real-World Architecture

```
Routers → Streaming Telemetry → Kafka → RL Engine → SDN Controller → Update Link Weights
```

Used in:

* Google B4 WAN
* Facebook Express Backbone
* Microsoft WAN

Instead of static OSPF:
They use:

* Centralized SDN controller
* RL / optimization engines
* Dynamic path programming

---

# 🧠 Why This Is Powerful

Traditional OSPF:

* Reactive
* Static
* Local view

RL-Based Routing:

* Proactive
* Global view
* Learns long-term optimal routing

---

# 🔥 Interview Answer (Amazon NDE Level)

You can say:

> Instead of static OSPF cost tuning, we can model link selection as a reinforcement learning problem where the state represents link utilization and latency, the action represents cost adjustment, and reward is based on minimized congestion and delay. The agent learns optimal cost values dynamically, effectively replacing static OSPF metrics with adaptive routing intelligence.

That’s a very strong systems-level answer.

---

# 🚀 If you want next:

* Full mini SDN + RL simulation
* Multi-agent RL for large ISP backbone
* How to integrate RL with BGP path selection
* Production-grade RL using Stable-Baselines3
* How hyperscalers actually implement this

Tell me how deep you want to go.


In [14]:
import numpy as np
import random

# States represent congestion level
# 0 = low congestion
# 1 = medium congestion
# 2 = high congestion
states = [0, 1, 2]

# Actions: 0 = Path A-B-D, 1 = Path A-C-D
actions = [0, 1]

# Initialize Q-table
q_table = np.zeros((len(states), len(actions)))

alpha = 0.8
gamma = 0.9
epsilon = 0.1
episodes = 1000

for episode in range(episodes):

    state = random.choice(states)

    # epsilon-greedy
    if random.uniform(0,1) < epsilon:
        action = random.choice(actions)
    else:
        action = np.argmax(q_table[state])

    # Simulated reward logic
    # Suppose path 0 performs badly during congestion
    if state == 2 and action == 0:
        reward = -10
    else:
        reward = 10 - state*2

    next_state = random.choice(states)

    # Q-learning update
    q_table[state, action] = q_table[state, action] + alpha * (
        reward + gamma * np.max(q_table[next_state]) - q_table[state, action]
    )

print("Learned Q-table:")
print(q_table)

Learned Q-table:
[[83.6400434  81.750212  ]
 [80.39201537 80.8540584 ]
 [64.5489273  79.89476718]]


In [15]:
import numpy as np

# State: [utilization %, latency ms]
# Action: increase cost or decrease cost

q_table = np.zeros((3, 2))  # 3 utilization states, 2 actions

def get_state(util):
    if util < 40:
        return 0
    elif util < 70:
        return 1
    else:
        return 2

alpha = 0.7
gamma = 0.9
epsilon = 0.1

for _ in range(1000):
    utilization = np.random.randint(0,100)
    state = get_state(utilization)

    if np.random.rand() < epsilon:
        action = np.random.choice([0,1])
    else:
        action = np.argmax(q_table[state])

    # Reward logic
    if state == 2 and action == 0:  # high util but didn't increase cost
        reward = -20
    else:
        reward = 10

    next_state = get_state(np.random.randint(0,100))

    q_table[state, action] += alpha * (
        reward + gamma*np.max(q_table[next_state]) - q_table[state, action]
    )

print("Optimized OSPF Cost Policy:")
print(q_table)

Optimized OSPF Cost Policy:
[[99.99999999 99.99999984]
 [99.99999999 99.99999997]
 [69.99999997 99.99999999]]
